# HW2

## Q1

In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv")

ERROR! Session/line number was not unique in database. History logging moved to new session 5


In [3]:
data.head(5)

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [4]:
var_cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']

In [5]:
df_final = data[var_cols]

In [6]:
df_final.head(5)

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


In [7]:
df_final.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

## Q2

In [8]:
hrs_median = df_final['horsepower'].median()
print(hrs_median)

254.0


## Q3

In [9]:
n = len(df_final)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df_final.iloc[idx[:n_train]]
df_val = df_final.iloc[idx[n_train:n_train + n_val]]
df_test = df_final.iloc[idx[n_train + n_val:]]

In [10]:
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [11]:
base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

X_train = df_train[base].values

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
y_test = df_test.fuel_efficiency_mpg.values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [12]:
df_train.head()

,engine_displacement,horsepower,vehicle_weight,model_year
0,1900,239.0,3790,2015
1,2000,224.0,4380,1992
2,2330,286.0,4090,2022
3,2300,NaN,4150,1997
4,2340,269.0,4400,2001


In [13]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [14]:
df_train[base].isnull().sum()

engine_displacement      0
horsepower             538
vehicle_weight           0
model_year               0
dtype: int64

In [15]:
hrs_mean = df_train['horsepower'].mean()
print(hrs_mean)

254.46338337605272


In [16]:
X_train_fillzeros = df_train[base].fillna(0).values
X_val_fillzeros = df_val[base].fillna(0).values
X_train_fillmedian = df_train[base].fillna(hrs_mean).values
X_val_fillmedian = df_val[base].fillna(hrs_mean).values

In [17]:
w0_zeros, w_zeros = train_linear_regression(X_train_fillzeros, y_train)
y_pred_zeros = w0_zeros + X_val_fillzeros.dot(w_zeros)

print(w0_zeros)
print(w_zeros)
print(y_pred_zeros)

-153.8849618823138
[-1.51505520e-03 -4.74024794e-06 -3.95418397e-03  1.02146785e-01]
[31.64502134 29.86572978 31.58838899 ... 30.53133298 30.75085098
 31.62912778]


In [18]:
w0_median, w_median = train_linear_regression(X_train_fillmedian, y_train)
y_pred_median = w0_median + X_val_fillmedian.dot(w_median)

print(w0_median)
print(w_median)
print(y_pred_median)

-153.29645639114293
[-0.00102386 -0.00648622 -0.00388828  0.10197897]
[31.54288381 29.92505122 31.56459523 ... 30.54408311 30.85701236
 31.55313717]


In [19]:
def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

In [20]:
rmse(y_val,y_pred_zeros).round(3)

np.float64(2.205)

In [21]:
rmse(y_val,y_pred_median).round(3)

np.float64(2.202)

## Q4

In [22]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [23]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression_reg(X_train_fillzeros, y_train, r=r)

    y_pred = w0 + X_val_fillzeros.dot(w)
    score = rmse(y_val, y_pred)

    print(r, w0, score.round(4))

0 -153.8849618823138 2.2053
0.01 -148.3092124404723 2.2058
0.1 -111.83871039306318 2.2241
1 -32.331865964971264 2.3492
5 -7.7728522099734025 2.4094
10 -3.9871164160193766 2.4195
100 -0.40821964884790385 2.4292


## Q5

In [24]:
seed_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
rmse_seeds = []

for i in seed_list:
    
    np.random.seed(i)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train = df_final.iloc[idx[:n_train]]
    df_val = df_final.iloc[idx[n_train:n_train + n_val]]
    df_test = df_final.iloc[idx[n_train + n_val:]]

    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
    
    X_train = df_train[base].values
    
    y_train = df_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values
    y_test = df_test.fuel_efficiency_mpg.values
    
    del df_train['fuel_efficiency_mpg']
    del df_val['fuel_efficiency_mpg']
    del df_test['fuel_efficiency_mpg']

    X_train_fillzeros = df_train[base].fillna(0).values
    X_val_fillzeros = df_val[base].fillna(0).values

    w0_zeros, w_zeros = train_linear_regression(X_train_fillzeros, y_train)
    y_pred_zeros = w0_zeros + X_val_fillzeros.dot(w_zeros)

    rmse_seeds.append(rmse(y_val,y_pred_zeros))

In [25]:
rmse_seeds

[np.float64(2.239204143046126),
 np.float64(2.2021128210838845),
 np.float64(2.1634197787530947),
 np.float64(2.2043588768539224),
 np.float64(2.2018800943311554),
 np.float64(2.2457851509085134),
 np.float64(2.2697212079524043),
 np.float64(2.190794599933443),
 np.float64(2.216611201686444),
 np.float64(2.2070365928178957)]

In [26]:
round(np.std(rmse_seeds),3)

np.float64(0.029)

## Q6

In [29]:
seed_list = [9]

for i in seed_list:
    
    np.random.seed(i)
    idx = np.arange(n)
    np.random.shuffle(idx)
    
    df_train = df_final.iloc[idx[:n_train]]
    df_val = df_final.iloc[idx[n_train:n_train + n_val]]
    df_test = df_final.iloc[idx[n_train + n_val:]]

    df_train = df_train.reset_index(drop=True)
    df_val = df_val.reset_index(drop=True)
    df_test = df_test.reset_index(drop=True)

    base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
    
    df_full_train = pd.concat([df_train,df_val])
    
    X_train = df_full_train[base].values
    X_test = df_test[base].fillna(0).values
    
    y_train = df_full_train.fuel_efficiency_mpg.values
    y_val = df_val.fuel_efficiency_mpg.values
    y_test = df_test.fuel_efficiency_mpg.values
    
    del df_full_train['fuel_efficiency_mpg']
    del df_val['fuel_efficiency_mpg']
    del df_test['fuel_efficiency_mpg']

    X_train_fillzeros = df_full_train[base].fillna(0).values
    X_val_fillzeros = df_val[base].fillna(0).values

    w0_zeros, w_zeros = train_linear_regression_reg(X_train_fillzeros, y_train, r=0.001)
    y_pred_zeros = w0_zeros + X_test.dot(w_zeros)

    print(rmse(y_test,y_pred_zeros))


2.235827747191731
